# Quick start: train your first nowcasting model with `mlcast`

This notebook is a beginner-friendly, end-to-end walkthrough of the MLCast
stack. In about ten minutes on a laptop CPU you will:

1. open the [mlcast-datasets](https://github.com/mlcast-community/mlcast-datasets)
   catalog and pick a radar dataset,
2. download one rainy day of it to your machine,
3. configure a small ConvGRU ensemble model with the
   [mlcast](https://github.com/mlcast-community/mlcast) Python API,
4. train it for a few minutes on CPU, and
5. make (and look at) a probabilistic precipitation nowcast.

The model trained here is deliberately tiny and undertrained, so its forecasts
will not be good. The point is to see every piece working together; the
[last section](#next-steps) explains how to scale up.

```{note}
Requires Python ≥ 3.11 and about 150 MB of free disk space. The data comes
from a public, anonymous-access object store, so no credentials are needed.
```

## 1. Installation

Install `mlcast`, the dataset catalog and the plotting dependencies into a
fresh environment:

```bash
pip install mlcast mlcast-datasets cartopy matplotlib
```

`mlcast` is in rapid development, so if you plan to change the code, clone the
repository and install it with `uv` instead, as described in
[Usage](usage.md#installation). This notebook was run against the version
printed below.

In [ ]:
import warnings
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import fiddle as fdl
import mlcast_datasets
import numpy as np
import torch
import xarray as xr
from fiddle import printing as fdp

import mlcast
from mlcast.config import (
    set_variables,
    training_experiment,
    use_random_sampler,
    validate_config,
)

# keep the rendered output readable: mlcast warns that it treats rainfall
# flux like rainfall rate when converting to reflectivity (see section 4)
warnings.filterwarnings("ignore", category=UserWarning)

print("mlcast version:", mlcast.__version__)

## 2. Open the dataset catalog

All MLCast datasets are published as cloud-optimised Zarr stores and listed in
an [Intake](https://intake.readthedocs.io/) catalog. Opening a dataset is
*lazy*: only metadata is read, so this is fast even though the full archives
are terabytes in size.

In [ ]:
cat = mlcast_datasets.open_catalog()
list(cat.precipitation)

We use the Belgian RMI **RADCLIM** radar–rain-gauge merged rain rate
(`be_rmi_radclim_mfb_5min`): 5-minute steps on a 700 × 700 km grid, small
enough to work with on a laptop. See the
[dataset page](datasets/be_rmi_precipitation.ipynb) for details.

In [ ]:
ds_remote = cat.precipitation.be_rmi_radclim_mfb_5min.to_dask()
ds_remote

`mlcast` finds variables by their
[CF standard name](https://cfconventions.org/Data/cf-standard-names/current/build/cf-standard-name-table.html)
rather than by variable name, so the same training code works across
datasets. Here the rain rate is stored as `rain_rate` with standard name
`rainfall_flux`:

In [ ]:
var_name = "rain_rate"
standard_name = ds_remote[var_name].attrs["standard_name"]
print(f"{var_name}: standard_name={standard_name!r}, units={ds_remote[var_name].attrs['units']!r}")

## 3. Download one rainy day

Training reads many small random crops, which is much faster from local disk
than from the object store. We download **14 July 2021**, the day of the
severe flooding in eastern Belgium (Journée et al., 2023), which gives
288 five-minute frames with plenty of rain. The download is roughly 100 MB
and is skipped if the file already exists.

In [ ]:
zarr_path = Path("data/quickstart_be_rmi_2021-07-14.zarr")

if not zarr_path.exists():
    ds_day = ds_remote[[var_name]].sel(time=slice("2021-07-14T00:00", "2021-07-14T23:55"))
    # drop the remote store's chunk/compression settings so xarray picks
    # sensible ones for the local copy
    for name in ds_day.variables:
        ds_day[name].encoding.clear()
    ds_day.to_zarr(zarr_path, mode="w")

ds = xr.open_zarr(zarr_path)
ds

A quick look at the data: three frames, one hour apart, during the afternoon.

In [ ]:
grid_mapping = ds[var_name].attrs["grid_mapping"]
data_crs = ccrs.Projection(ds[grid_mapping].attrs["crs_wkt"])


def plot_frames(da: xr.DataArray, **kwargs) -> xr.plot.FacetGrid:
    """Plot rain-rate frames on a map, one panel per entry along ``time``.

    Parameters
    ----------
    da : xr.DataArray
        Rain rate with dimensions ``(time, y, x)``.
    **kwargs
        Forwarded to ``xarray.DataArray.plot``.

    Returns
    -------
    xr.plot.FacetGrid
        The facet grid holding the map axes.
    """
    g = da.plot(
        transform=data_crs,
        subplot_kws=dict(projection=data_crs),
        cmap="Blues",
        vmin=0,
        vmax=10,
        cbar_kwargs=dict(label="rain rate [mm/h]", shrink=0.8),
        **kwargs,
    )
    for ax in g.axs.flat:
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    return g


plot_frames(ds[var_name].sel(time=["2021-07-14T14:00", "2021-07-14T15:00", "2021-07-14T16:00"]), col="time");

## 4. Configure the experiment

`mlcast` describes a whole experiment (dataset, data module, network,
Lightning module and trainer) as a single [Fiddle](https://github.com/google/fiddle)
config graph. We start from the default `training_experiment` and modify it;
nothing is built until we call `fdl.build`. The same changes can be made from
the command line (see [Next steps](#next-steps)).

| Change | Why |
|---|---|
| `use_random_sampler` | draw random crops on the fly instead of reading a precomputed sampling index |
| `set_variables(["rainfall_flux"])` | select the variable by CF standard name and set the network's input channels to match |
| `zarr_path` | point at our local copy |
| `forecast_steps=6` | predict 30 minutes ahead from the past 30 minutes (`input_steps=6`) |
| `width=height=128`, `epoch_size=256` | small crops and short epochs so training is quick on CPU |
| `batch_size=4`, `num_workers=0`, `pin_memory=False` | CPU-friendly data loading |
| `network.num_blocks=3`, `optimizer.lr=1e-3` | a shallower network (~250 k instead of ~64 M parameters) and a higher learning rate, so something is learnt in a few minutes |
| `loss_params={"temporal_lambda": 0.0}` | turn off the CRPS temporal-smoothness penalty (works around a bug in `mlcast` v0.1.0) |
| `trainer.max_epochs=8`, `accelerator="cpu"` | eight short epochs (about four minutes on a laptop) on CPU |

By default the data is split in time: the first 70 % of the day for training,
the next 15 % for validation and the last 15 % for testing.

In [ ]:
cfg = training_experiment.as_buildable()

use_random_sampler(cfg)
set_variables(cfg, standard_names=[standard_name])

dataset_factory = cfg.data.dataset_factory
dataset_factory.zarr_path = str(zarr_path)
dataset_factory.forecast_steps = 6
dataset_factory.width = 128
dataset_factory.height = 128
dataset_factory.epoch_size = 256

cfg.data.batch_size = 4
cfg.data.num_workers = 0
cfg.data.pin_memory = False

cfg.pl_module.network.num_blocks = 3
cfg.pl_module.optimizer.lr = 1e-3

# mlcast v0.1.0 computes a zero masked CRPS loss when the temporal penalty is
# enabled, so switch the penalty off
cfg.pl_module.loss_params = {"temporal_lambda": 0.0}

cfg.trainer.max_epochs = 8
cfg.trainer.accelerator = "cpu"
cfg.trainer.log_every_n_steps = 8
cfg.trainer.enable_progress_bar = False

validate_config(cfg)

# print the parameters we changed (and the rest of the dataset settings)
for key, value in fdp.as_dict_flattened(cfg).items():
    if key.startswith(
        (
            "data.",
            "trainer.max_epochs",
            "trainer.accelerator",
            "pl_module.network.",
            "pl_module.loss",
            "pl_module.optimizer.lr",
        )
    ):
        print(f"{key:45s} {value}")

## 5. Train

`fdl.build` turns the config into real objects: a PyTorch Lightning
`Trainer`, a `NowcastLightningModule` wrapping the ConvGRU network, and the
data module. The model is trained with the CRPS loss on an ensemble of
two members, so it learns to produce a *distribution* of possible futures
rather than a single blurry mean. Metrics are written to TensorBoard under
`logs/` (`tensorboard --logdir logs`).

```{tip}
`mlcast.config.train_from_config(cfg)` does all of this in one call (and also
saves the config next to the logs). We call the steps separately here so we
can keep the trained model in memory for the next section.
```

In [ ]:
experiment = fdl.build(cfg)
experiment.trainer.fit(experiment.pl_module, datamodule=experiment.data)

print("Log directory:", experiment.trainer.logger.log_dir)
print({k: round(float(v), 4) for k, v in experiment.trainer.callback_metrics.items()})

## 6. Make a nowcast

`NowcastLightningModule.predict` takes the most recent observed frames in
physical units (here mm/h), handles normalisation, and returns
`(ensemble_member, lead_time, y, x)`. The network is fully convolutional, so
although it was trained on 128 × 128 crops we can run it on the whole
700 × 700 domain.

We start the forecast at 20:30, inside the held-out test period, so the model
has not seen these frames during training.

In [ ]:
model = experiment.pl_module
input_steps = dataset_factory.input_steps
forecast_steps = dataset_factory.forecast_steps

t0 = ds.time.to_index().get_loc(np.datetime64("2021-07-14T20:30"))
da_past = ds[var_name].isel(time=slice(t0 - input_steps + 1, t0 + 1)).load()
da_future = ds[var_name].isel(time=slice(t0 + 1, t0 + 1 + forecast_steps)).load()

torch.manual_seed(0)
preds = model.predict(
    torch.from_numpy(da_past.values),
    forecast_steps=forecast_steps,
    ensemble_size=2,
    standard_name=standard_name,
)
da_forecast = xr.DataArray(
    preds,
    dims=("member", "time", "y", "x"),
    coords=dict(member=[1, 2], time=da_future.time, y=ds.y, x=ds.x),
)
da_forecast.shape

Rows show the observed rain (top) and the two ensemble members, at lead times
of 10, 20 and 30 minutes. After a few minutes of training the model already
puts rain in the right places, but the fields are blurry, the small-scale
structure is gone and the two members barely differ. A properly trained model
keeps the rain structures sharp and uses the members to show where the
forecast is uncertain.

In [ ]:
lead_idx = [1, 3, 5]
da_compare = xr.concat(
    [
        da_future.isel(time=lead_idx).expand_dims(source=["observed"]),
        da_forecast.isel(time=lead_idx).rename(member="source").assign_coords(source=["member 1", "member 2"]),
    ],
    dim="source",
)
da_compare = da_compare.assign_coords(lead_time=("time", [f"+{5 * (i + 1)} min" for i in lead_idx])).swap_dims(
    time="lead_time"
)
plot_frames(da_compare, col="lead_time", row="source");

(next-steps)=
## 7. Next steps

**Use the command line.** Every change above can be made with `--config`
flags instead, and the resolved config is saved with the logs so runs are
reproducible:

```bash
mlcast train \
    --config fiddler:use_random_sampler \
    --config "fiddler:set_variables(standard_names=['rainfall_flux'])" \
    --config set:data.dataset_factory.zarr_path=data/quickstart_be_rmi_2021-07-14.zarr \
    --config set:data.dataset_factory.forecast_steps=6 \
    --config set:data.dataset_factory.width=128 \
    --config set:data.dataset_factory.height=128 \
    --config set:data.dataset_factory.epoch_size=256 \
    --config set:data.batch_size=4 \
    --config set:data.num_workers=0 \
    --config set:pl_module.network.num_blocks=3 \
    --config set:pl_module.optimizer.lr=0.001 \
    --config "set:pl_module.loss_params={'temporal_lambda': 0.0}" \
    --config set:trainer.max_epochs=8 \
    --config set:trainer.accelerator=cpu
```

Add `--print_config_and_exit` to check the resolved config without training.

**Scale up.**

- *More data*: train on years rather than a day. Read directly from the object
  store with the `use_anon_s3_dataset` fiddler, or download a larger slice.
- *Better samples*: build a precomputed sampling index that favours rainy
  crops instead of sampling at random; see
  [Sampling training-ready data](data.md#sampling-training-ready-data).
- *GPU*: install with `uv sync --extra gpu-cu128` (or `gpu-cu130`) and drop
  the `accelerator="cpu"` override; use the default 256 × 256 crops,
  batch size and epoch count.
- *Track experiments*: switch to MLflow with `fiddler:use_mlflow_logger`.

**Go further.**

- [Usage](usage.md): the full configuration model, CLI and Python API
- [Model architecture](model-architecture.md): how the ConvGRU encoder–decoder works
- [Examples](examples.md): plug in your own network architecture
- [Datasets](data.md): all datasets in the catalog

**Reference:** Journée, M., Goudenhoofdt, E., Vannitsem, S. & Delobbe, L.
(2023). Quantitative rainfall analysis of the 2021 mid-July flood event in
Belgium. *Hydrol. Earth Syst. Sci.* 27(17), 3169–3189.
<https://doi.org/10.5194/hess-27-3169-2023>